In [1]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings,ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
import os,glob

from sqlalchemy.testing.suite.test_reflection import metadata

load_dotenv()

#----加载----
def load_text(path:str):
    with open(path,encoding="utf-8") as f:
        text=f.read()
    return Document(page_content=text,metadata={"source":path})

all_docs = []
for f in glob.glob("data/*.md"):
    all_docs.append(load_text(f))
print("文档数:",len(all_docs))

#----分块----
splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
chunks = splitter.split_documents(all_docs)
print("分块数:",len(chunks))

#----向量化+入库----
embeddings = OpenAIEmbeddings(
    model="BAAI/bge-m3",
    api_key=os.getenv("SILICONFLOW_API_KEY"),
    base_url="https://api.siliconflow.cn/v1"
)
vectorstore = Chroma(
    collection_name="my_notes",
    embedding_function=embeddings,
    persist_directory="chroma_db"
)
if vectorstore._collection.count() == 0:
    vectorstore.add_documents(chunks)

retriever = vectorstore.as_retriever(search_kwargs={"k":3})

#----检索+拼prompt+回答----
def ask(question:str)->str:
    docs = retriever.invoke(question)  #检索
    context = "\n\n".join(d.page_content for d in docs)  #拼上下文

    prompt = ChatPromptTemplate.from_messages([
        ("system","你是一个知识库问答助手。只根据下面的资料回答，资料里没有就说不知道。\n\n资料：\n{context}"),
        ("user","{question}"),
    ])
    llm = ChatOpenAI(
        model="deepseek-v4-flash",
        api_key=os.getenv("DEEPSEEK_API_KEY"),
        base_url="https://api.deepseek.com",
        temperature=0.3,
        max_tokens=500
    )

    chain = prompt|llm|StrOutputParser()

    return chain.invoke({"context":context,"question":question})  #回答


文档数: 3
分块数: 186


#检索质量调试

In [2]:
#先看检索到了什么
def show_retrieval(question:str):
    docs = retriever.invoke(question)
    for i,d in enumerate(docs,1):
        print(f"[{i}] {d.metadata.get('source')}: {d.page_content[:80]}")
    return docs

show_retrieval("什么是RAG")

[1] data\rag_notes_1.md: ### 6.3 API 设计
| 方法 | 路径 | 作用 |
|---|---|---|
| POST | /api/upload | 上传文件，返回 fil
[2] data\第一周实操指南_Day1-9.md: ### 1.8 把第一个脚本推上 GitHub（🔧）
1. GitHub 网页 → New repository → 名字 hello-llm → 勾选 REA
[3] data\第二周实操指南_Day10-16.md: ```python
from pydantic import BaseModel, Field

class Resume(BaseModel):
    na


[Document(id='1ca7de74-84ca-47bd-a3ef-52c862087455', metadata={'source': 'data\\rag_notes_1.md'}, page_content='### 6.3 API 设计\n| 方法 | 路径 | 作用 |\n|---|---|---|\n| POST | /api/upload | 上传文件，返回 file_id |\n| POST | /api/ingest | 触发解析入库（可合并进 upload） |\n| GET | /api/documents | 列出已入库文档 |\n| DELETE | /api/documents/{id} | 删除文档与对应向量 |\n| POST | /api/chat | 多轮问答（SSE 流式返回 回答 + sources） |\n| GET | /health | 健康检查（部署用） |'),
 Document(id='c23c77b8-2fbb-4dcb-a663-b6096bae6dcf', metadata={'source': 'data\\第一周实操指南_Day1-9.md'}, page_content='### 1.8 把第一个脚本推上 GitHub（🔧）\n1. GitHub 网页 → New repository → 名字 hello-llm → 勾选 README → Create\n2. 本地执行：\n```powershell\ncd C:\\Users\\17357\\Documents\\Codex\\2026-07-31\\26-ai-langchain-llamaindex-chroma-deepseek\\work\\hello-llm\ngit init\ngit add .\ngit commit -m "第一个 DeepSeek 调用"\ngit branch -M main\ngit remote add origin git@github.com:你的用户名/hello-llm.git\ngit push -u origin main\n```\n3. 刷新 GitHub 页面：能看到 hello_deepseek.py，.env 不在里面（正确！）'),
 Document(id='9b8b0

In [3]:
#调参实验：chunk_size对比
for s in [200,500,1000]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=s,chunk_overlap=50)
    chunks = splitter.split_documents(all_docs)
    vs = Chroma(collection_name=f"exp_{s}",embedding_function=embeddings,persist_directory="chroma_db")
    if vs._collection.count() == 0:
        vs.add_documents(chunks)
    hits = vs.similarity_search("什么是RAG?",k=3)
    print(f"chunk_size={s}:命中来源{[h.metadata.get('source')for h in hits]}")

chunk_size=200:命中来源['data\\第一周实操指南_Day1-9.md', 'data\\第一周实操指南_Day1-9.md', 'data\\第一周实操指南_Day1-9.md']
chunk_size=500:命中来源['data\\第一周实操指南_Day1-9.md', 'data\\rag_notes_1.md', 'data\\第二周实操指南_Day10-16.md']
chunk_size=1000:命中来源['data\\第一周实操指南_Day1-9.md', 'data\\第一周实操指南_Day1-9.md', 'data\\第二周实操指南_Day10-16.md']
